# 🔑 重みキー × mapper

全キーに mapper を適用し、未変換・TP=2 分割可否を出す。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

def find_root():
    """dataset のマウント位置は 2 通りあるので config.json を探す。"""
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

import json
import struct

ROOT = find_root()
TP = 2

print("\n=== どのクラスが使われるか（config.architectures）===")
cfg = json.load(open(ROOT + "/config.json"))
print("  architectures:", cfg.get("architectures"), "| model_type:", cfg.get("model_type"))

print("\n=== mapper 定義（実クラスごと）===")
for f in ("qwen3_5.py", "qwen3_vl.py", "qwen3_5_mtp.py"):
    t = (root / "model_executor/models" / f).read_text(errors="replace")
    for i, l in enumerate(t.splitlines(), 1):
        if "hf_to_vllm_mapper" in l or "orig_to_new_" in l:
            print(f"  {f}:{i}: {l.strip()[:200]}")

print("\n=== 生キーの取得 ===")


def header(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(n).decode("utf-8", "replace")), 8 + n


h_main, base_main = header(ROOT + "/model.safetensors")
h_mtp, _ = header(ROOT + "/model-mtp.safetensors")
keys_main = [k for k in h_main if k != "__metadata__"]
keys_mtp = [k for k in h_mtp if k != "__metadata__"]
print(f"  model.safetensors: {len(keys_main)} tensors")
print(f"  model-mtp.safetensors: {len(keys_mtp)} tensors")
print("  dtype 内訳:", {d: sum(1 for v in h_main.values()
                              if isinstance(v, dict) and v.get("dtype") == d)
                       for d in {v.get("dtype") for v in h_main.values()
                                 if isinstance(v, dict)}})


def apply_map(keys, prefix_map):
    out, dropped = [], []
    for k in keys:
        for pre, new in prefix_map.items():
            if k.startswith(pre):
                if new is None:
                    dropped.append(k)
                else:
                    out.append(new + k[len(pre):])
                break
        else:
            out.append(k)          # 一致なしはそのまま
    return out, dropped


TEXT_MAP = {"model.language_model.": "model.", "mtp.": None}
VL_MAP = {"model.visual.": "visual.", "lm_head.": "language_model.lm_head.",
          "model.language_model.": "language_model.model.", "mtp.": None}
DRAFT_MAP = {"mtp.": "model."}

for name, m in (("text-only クラス", TEXT_MAP), ("VL クラス(実使用)", VL_MAP)):
    out, dropped = apply_map(keys_main, m)
    pref = sorted({".".join(k.split(".")[:2]) for k in out})
    print(f"\n--- {name}: {len(out)} 変換 / {len(dropped)} 破棄 ---")
    print("  prefix:", pref[:8])
    bad = [k for k in out if k.startswith(("model.visual.", "model.language_model.",
                                           "lm_head."))]
    print("  変換漏れ:", len(bad), bad[:3])

out_vl, dropped_vl = apply_map(keys_main, VL_MAP)
print("\n--- VL mapper の内訳 ---")
for pre in ("visual.", "language_model.model.", "language_model.lm_head."):
    n = sum(1 for k in out_vl if k.startswith(pre))
    print(f"  {pre:26s} {n}")
print("  想定外:", [k for k in out_vl if not k.startswith(
    ("visual.", "language_model.model.", "language_model.lm_head."))][:4])
print("  dropped:", dropped_vl, "（main に mtp.* は無いのが正）")

print("\n=== vision の stacked mapping（q/k/v -> qkv）===")
st = [k for k in out_vl if k.startswith("visual.") and ".attn.q." in k]
print("  visual attn.q の例:", st[:2], "件数:", len(st))
print("  → vllm は attn.q/k/v を qkv にまとめる（orig_to_new_stacked）")

print("\n=== ignore 305 件との突き合わせ（変換後）===")
qc = cfg.get("quantization_config") or {}
ignore = list(qc.get("ignore") or [])
allk = out_vl
hit = [k for k in allk if any(k.startswith(i) for i in ignore)]
print("  ignore:", len(ignore), "| 一致した重み:", len(hit), hit[:3])
quant = [k for k in allk if k.endswith(("weight_packed", "weight_scale",
                                         "weight_shape", "weight_zero_point"))]
print("  量子化系キー:", len(quant))
print("  量子化系なのに ignore 一致:", len([k for k in quant
                                     if any(k.startswith(i) for i in ignore)]))
print("  ignore されるのに量子化系:", len([k for k in hit if k in quant]))

print("\n=== draft mapper ===")
k2, d2 = apply_map(keys_mtp, DRAFT_MAP)
print(f"  {len(k2)} 変換 / {len(d2)} 破棄")
print("  例:", k2[:4])
print("  想定外（model. で始まらない）:", [k for k in k2 if not k.startswith("model.")][:3])

print("\n=== weight_shape の実体（dtype で unpack を変える）===")
FMT = {"I32": ("<2i", 8), "I64": ("<2q", 16), "U32": ("<2I", 8), "U64": ("<2Q", 16)}
shapes = {}
with open(ROOT + "/model.safetensors", "rb") as f:
    for k in keys_main:
        if not k.endswith("weight_shape"):
            continue
        meta = h_main[k]
        a, b = meta["data_offsets"]
        fmt, need = FMT.get(meta.get("dtype"), ("<2i", 8))
        if b - a < need:
            print("  短すぎる:", k, meta.get("dtype"), b - a)
            continue
        f.seek(base_main + a)
        shapes[k] = struct.unpack(fmt, f.read(need))
print("  dtype:", {h_main[k].get("dtype") for k in shapes}, "| 件数:", len(shapes))
print("  例:", list(shapes.items())[:3])

odd = [(k, s) for k, s in shapes.items() if s[0] % TP or s[1] % TP]
print(f"  TP={TP} で割れない shape:", len(odd), odd[:5])
mx = max(shapes.values(), key=lambda s: s[0] * s[1]) if shapes else None
print("  最大 shape:", mx)

print("\n=== 代表レイヤ ===")
for k in sorted(shapes):
    if "layers.0." in k and any(x in k for x in ("qkv", "z.weight", "o_proj",
                                                 "gate_up", "down_proj")):
        print(f"  {k.split('.language_model.')[-1]}: {shapes[k]}")
